# Reflection workflow validation

This notebook runs the proposal-inspection integration test and builds isolated experience cases through the real CLI. The cases test discovery and evidence handling. They do not measure production repair success or retrieval benefit.

The fixture contains six checked cases, one unchecked problem, a pending consolidation proposal, an existing skill, and a generated rules file with a canonical source. A separate agent trial uses the fixture without reading its seed definitions.

In [1]:
import hashlib
import json
import os
import platform
import shlex
import subprocess
from datetime import UTC, datetime
from pathlib import Path

from IPython.display import display as show

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "Cargo.toml").is_file()
)
STARTED_AT = datetime.now(UTC)
CHECK_ENV = os.environ | {
    "RUSTC_WRAPPER": "",
    "CARGO_BUILD_JOBS": "8",
    "TMPDIR": "/var/tmp",
}


def run_command(command: list[str]) -> str:
    """Run a validation command and retain its exact invocation and output."""
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=CHECK_ENV,
        check=True,
        capture_output=True,
        text=True,
        timeout=900,
    )
    print(f"$ {shlex.join(command)}")
    print(completed.stdout)
    return completed.stdout


show({"started_at_utc": STARTED_AT.isoformat(), "host": platform.node()})

{'started_at_utc': '2026-09-18T20:58:34.706411+00:00', 'host': 'nelli-gpu'}

## Proposal inspection

The integration test exercises the CLI against a real persistent store: scope filtering before the limit, inspection with an active writer, hidden candidates in normal retrieval, wrong-scope rejection, and accept/reject through a warm worker.

In [2]:
inspection_output = run_command(
    [
        "cargo",
        "+1.98.0",
        "test",
        "-p",
        "memd",
        "--test",
        "consolidate_review_cli",
        "--",
        "--nocapture",
    ]
)

$ cargo +1.98.0 test -p memd --test consolidate_review_cli -- --nocapture

running 1 test
test review_cli_is_scoped_read_only_and_routes_decisions_through_worker ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.66s




## Checked-case discovery

The fixture runs a failing configuration check, changes the file, runs a passing check against unchanged evidence, then records a conditional lesson. It does this for six independent problems. Two queue cases recommend different values under the same declared conditions. Their passing checks do not resolve that conflict.

In [3]:
fixture = json.loads(
    run_command(["bash", "evals/bench/reflection/seed.sh", "target/debug/memd"])
)
fixture_root = Path(fixture["fixture_root"])
search = json.loads((fixture_root / "check-search.json").read_text(encoding="utf-8"))
inspection = json.loads((fixture_root / "inspection.json").read_text(encoding="utf-8"))
show(
    [
        {
            "task_id": hit["artifact"]["task_id"],
            "status": hit["artifact"]["status"],
            "project_id": hit["project_id"],
        }
        for hit in search["results"]
    ]
)
assert len(search["results"]) == 6
assert len(fixture["cases"]) == 6
assert inspection["run"]["state"] == "validated"
assert inspection["candidates"][0]["candidate"]["payload"]["status"] == "candidate"
assert inspection["sources"][0]["payload_missing"] is False

$ bash evals/bench/reflection/seed.sh target/debug/memd
{
  "fixture_root": "/var/tmp/memd-reflection.0enJvD",
  "binary": "/home/fschulz/dev/memd-worktrees/experience-memory/target/debug/memd",
  "tenant_id": "reflection_trial",
  "project_id": "demo",
  "run_id": "01a0b650-ad41-7e21-8e6f-7958c9cee0ae",
  "cases": [
    {
      "label": "manifest_a",
      "problem_id": "01a0b650-8a64-7e21-b30d-d26b32210597",
      "check_id": "01a0b650-8d68-7730-b782-f1bbd13ecdd7",
      "lesson_id": "01a0b650-8e3c-7290-aaae-f86bfa8f1a54"
    },
    {
      "label": "manifest_b",
      "problem_id": "01a0b650-8f12-71f0-b2f8-5d93205000a7",
      "check_id": "01a0b650-928d-7cd2-a014-1220e2115c2f",
      "lesson_id": "01a0b650-936d-74f2-9a9c-96d46bf08c00"
    },
    {
      "label": "cache_a",
      "problem_id": "01a0b650-9454-77c0-bbad-2138c9a7814a",
      "check_id": "01a0b650-9808-7303-8751-9028d1c26603",
      "lesson_id": "01a0b650-98f3-7060-8f09-78de9fce4594"
    },
    {
      "label": "cache_b"

[{'task_id': 'experience-01a0b650-a56b-7790-83d0-46ff1ea85e45',
  'status': 'passed',
  'project_id': 'demo'},
 {'task_id': 'experience-01a0b650-9f7f-7ae3-a508-2dfa9d8973a8',
  'status': 'passed',
  'project_id': 'demo'},
 {'task_id': 'experience-01a0b650-99d2-7ee1-b2de-372d3ccc1f0b',
  'status': 'passed',
  'project_id': 'demo'},
 {'task_id': 'experience-01a0b650-9454-77c0-bbad-212d98528410',
  'status': 'passed',
  'project_id': 'demo'},
 {'task_id': 'experience-01a0b650-8f12-71f0-b2f8-5d812829d19f',
  'status': 'passed',
  'project_id': 'demo'},
 {'task_id': 'experience-01a0b650-8a64-7e21-b30d-d2506e207fcf',
  'status': 'passed',
  'project_id': 'demo'}]

## Provenance and interpretation

The checks establish that the documented CLI path finds checked artifacts and inspects a pending proposal without accepting it. The agent must still judge where guidance belongs and whether its evidence supports it. The handoff links the independent trial report and its review. Use the source and binary hashes below to identify this build; its package version remains 1.7.1.

In [4]:
source_paths = [
    "crates/memd/src/cli/args.rs",
    "crates/memd/src/cli/consolidate.rs",
    "crates/memd/src/cli/mod.rs",
    "crates/memd/src/cli/scope.rs",
    "crates/memd/src/cli/warm.rs",
    "crates/memd/src/store/metadata/sqlite/consolidation.rs",
    "crates/memd/tests/consolidate_review_cli.rs",
    "memd-skill/SKILL.md",
    "memd-skill/references/reflection.md",
    "evals/bench/reflection/seed.sh",
]
source_hashes = {
    name: hashlib.sha256((REPO_ROOT / name).read_bytes()).hexdigest()
    for name in source_paths
}
with (REPO_ROOT / "target/debug/memd").open("rb") as binary:
    binary_hash = hashlib.file_digest(binary, "sha256").hexdigest()
provenance = {
    "git_revision": run_command(["git", "rev-parse", "HEAD"]).strip(),
    "git_status": run_command(["git", "status", "--short"]).splitlines(),
    "cargo": run_command(["cargo", "+1.98.0", "--version"]).strip(),
    "python": platform.python_version(),
    "binary_sha256": binary_hash,
    "source_sha256": source_hashes,
    "finished_at_utc": datetime.now(UTC).isoformat(),
}
show(provenance)

$ git rev-parse HEAD
8c7fb5bb587063bf160428c119614eed97e7105b

$ git status --short
 M crates/memd/src/cli/args.rs
 M crates/memd/src/cli/consolidate.rs
 M crates/memd/src/cli/mod.rs
 M crates/memd/src/cli/scope.rs
 M crates/memd/src/cli/warm.rs
 M crates/memd/src/store/metadata/sqlite/consolidation.rs
 M docs/agent-skill.md
 M docs/cli-reference.md
 M docs/self-improvement.md
 M memd-skill/README.md
 M memd-skill/SKILL.md
 M memd-skill/references/self-improvement.md
?? crates/memd/tests/consolidate_review_cli.rs
?? evals/bench/reflection/
?? memd-skill/references/reflection.md

$ cargo +1.98.0 --version
cargo 1.98.0 (797e8a9bc 2026-08-05)



{'git_revision': '8c7fb5bb587063bf160428c119614eed97e7105b',
 'git_status': [' M crates/memd/src/cli/args.rs',
  ' M crates/memd/src/cli/consolidate.rs',
  ' M crates/memd/src/cli/mod.rs',
  ' M crates/memd/src/cli/scope.rs',
  ' M crates/memd/src/cli/warm.rs',
  ' M crates/memd/src/store/metadata/sqlite/consolidation.rs',
  ' M docs/agent-skill.md',
  ' M docs/cli-reference.md',
  ' M docs/self-improvement.md',
  ' M memd-skill/README.md',
  ' M memd-skill/SKILL.md',
  ' M memd-skill/references/self-improvement.md',
  '?? crates/memd/tests/consolidate_review_cli.rs',
  '?? evals/bench/reflection/',
  '?? memd-skill/references/reflection.md'],
 'cargo': 'cargo 1.98.0 (797e8a9bc 2026-08-05)',
 'python': '3.12.13',
 'binary_sha256': '745cc59ece984aaf272a54d7931c26cf3cd510f7ea291c94f228269ab90a795b',
 'source_sha256': {'crates/memd/src/cli/args.rs': 'c4be1d21eba675772bf6809f7a456d04fa99ab651dc839f2f2aeef8aae30cf76',
  'crates/memd/src/cli/consolidate.rs': 'e449ea12cde857fd03e914ae66886062